# **1. Perkenalan Dataset**

Dataset yang digunakan adalah Customer Segmentation. Dataset berisi informasi umur, total gaji, total pengeluaran, frekuensi transaksi tahunan, dan status membership pelanggan. Tujuan eksperimen ini adalah menyiapkan data agar siap digunakan untuk melatih model klasifikasi status membership.

# **2. Import Library**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler


# **3. Memuat Dataset**

In [ ]:
raw_path = '../CustomerSegmentation_raw.csv'
df = pd.read_csv(raw_path)
df.head()


In [ ]:
df.info()


# **4. Exploratory Data Analysis (EDA)**

EDA dilakukan untuk memahami ukuran data, tipe data, missing value, duplikasi, distribusi numerik, serta distribusi target.

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
df.describe(include='all')


In [ ]:
df['Membership Status'].value_counts().plot(kind='bar', title='Distribusi Membership Status')
plt.xlabel('Membership Status')
plt.ylabel('Jumlah Pelanggan')
plt.tight_layout()
plt.show()


In [ ]:
numeric_columns = ['Age', 'Total Salary (IDR)', 'Total Spending (IDR)', 'Frequency (Yearly)']
df[numeric_columns].hist(figsize=(10, 8), bins=20)
plt.tight_layout()
plt.show()


# **5. Data Preprocessing**

Tahapan preprocessing yang digunakan: menghapus duplikasi, memastikan kolom numerik bertipe angka, menghapus missing value, encoding target, dan standardisasi fitur numerik.

In [ ]:
feature_columns = ['Age', 'Total Salary (IDR)', 'Total Spending (IDR)', 'Frequency (Yearly)']
target_column = 'Membership Status'

clean_df = df.drop_duplicates().copy()
clean_df = clean_df.dropna(subset=feature_columns + [target_column]).copy()

for column in feature_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors='coerce')
clean_df = clean_df.dropna(subset=feature_columns).copy()

label_encoder = LabelEncoder()
clean_df['membership_status_label'] = label_encoder.fit_transform(clean_df[target_column])

scaler = StandardScaler()
scaled_features = scaler.fit_transform(clean_df[feature_columns])
processed_df = pd.DataFrame(
    scaled_features,
    columns=['age_scaled', 'total_salary_idr_scaled', 'total_spending_idr_scaled', 'frequency_yearly_scaled']
)
processed_df['membership_status_label'] = clean_df['membership_status_label'].to_numpy()
processed_df['membership_status'] = clean_df[target_column].to_numpy()
processed_df.head()


In [ ]:
output_dir = 'CustomerSegmentation_preprocessing'
import os
os.makedirs(output_dir, exist_ok=True)
processed_df.to_csv(f'{output_dir}/customer_segmentation_preprocessed.csv', index=False)
pd.DataFrame({
    'membership_status': label_encoder.classes_,
    'membership_status_label': range(len(label_encoder.classes_))
}).to_csv(f'{output_dir}/label_mapping.csv', index=False)
processed_df.shape
